## Task 3, part 3 - Modelling of the second model 
In this part we want to train another model based on the training data used in the previous part but instead of bein deliberately simplistic, this model is supposed to outperform the previous model by using a more advanced feature engineering + training approach. 

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

In [2]:
# load back in the data prepared in Task3_01
DATA_DIR = "/home/ubuntu/data/frangieh"

pert_FC_selected = pd.read_pickle(f"{DATA_DIR}/task3_pert_FC_selected_50.pkl")
train_40 = pd.read_csv(f"{DATA_DIR}/task3_train_40.csv")["perturbation"].tolist()
test_10 = pd.read_csv(f"{DATA_DIR}/task3_test_10.csv")["perturbation"].tolist()
control_expression = pd.read_pickle(f"{DATA_DIR}/task3_control_expression.pkl")
# per-condition cell counts of the 50 selected perturbations, used below as Ridge sample weights
cell_counts = pd.read_csv(f"{DATA_DIR}/task3_cell_counts_selected_50.csv", index_col="perturbation")

# the 50 selected perturbations are simply the union of train and test
selected_50 = train_40 + test_10
conditions = pert_FC_selected.index.get_level_values("condition").unique().tolist()

pert_FC_selected.shape, control_expression.shape

((150, 2042), (56343, 2042))

## Feature vector per (gene, condition): plain summary statistics

Describe each of the 50 target genes by three directly interpretable numbers computed from its own expression in that condition's control cells: how highly it's normally expressed (mean), how much it fluctuates cell-to-cell (variance), and how often it reads zero (dropout rate). No PCA or correlation involved -- just descriptive statistics of the gene itself, computable for any gene including the 10 held out.

In [3]:
# feature vector per (gene, condition): [mean expression, variance, dropout rate], all from control cells
gene_features = {}
for cond in conditions:
    control_cond = control_expression.loc[cond]
    for gene in selected_50:
        expr = control_cond[gene].values
        mean_expr = expr.mean()
        var_expr = expr.var()
        dropout_rate = (expr == 0).mean()
        gene_features[(gene, cond)] = np.array([mean_expr, var_expr, dropout_rate])

gene_features[(selected_50[0], conditions[0])]

array([0.04366483, 0.03981897, 0.93457457])

## Reduce the target: PCA on the training fingerprints

Predicting all ~2042 genes' log2FC directly, from only 3 input features fit on 40 training genes, is too many outputs for too little data. Instead, fit PCA on the 40 training genes' fingerprints themselves (per condition) to find the main axes along which perturbation effects vary, and predict a gene's position along those axes instead of the full vector. This only ever uses training labels -- the 10 held-out genes are never involved in defining this space.

In [4]:
N_TARGET_PCS = 10

# per condition: fit PCA on the 40 training genes' fingerprints, and store each training gene's score
target_pca_by_condition = {}
target_scores = {}  # (gene, condition) -> score along the target PCs, training genes only
for cond in conditions:
    train_fingerprints = np.vstack([pert_FC_selected.loc[(g, cond)].values for g in train_40])
    pca = PCA(n_components=N_TARGET_PCS, random_state=42)
    scores = pca.fit_transform(train_fingerprints)
    target_pca_by_condition[cond] = pca
    for gene, score in zip(train_40, scores):
        target_scores[(gene, cond)] = score

# how much of the training fingerprints' variance these 10 PCs capture, per condition
{cond: target_pca_by_condition[cond].explained_variance_ratio_.sum() for cond in conditions}

{'Control': np.float32(0.61914194),
 'IFNγ': np.float32(0.6695988),
 'Co-culture': np.float32(0.76921785)}

## Ridge regression prediction

For a query gene in a given condition: standardize the 3 features (so the regularization penalty treats them fairly despite their very different scales), fit Ridge regression mapping features -> the 10 target-PC scores using the pool genes, predict the query's scores, then reconstruct the full fingerprint with that condition's target PCA. The pool never includes the query gene itself, so this works for both leave-one-out cross-validation and predicting the actual held-out genes.

In [5]:
def ridge_predict(query_gene, condition, alpha, pool_genes):
    """Predict a fingerprint via Ridge regression from gene features to target-PC scores."""
    # exclude the query gene itself from the pool used to fit the model
    fit_genes = [g for g in pool_genes if g != query_gene]

    X = np.vstack([gene_features[(g, condition)] for g in fit_genes])
    y = np.vstack([target_scores[(g, condition)] for g in fit_genes])

    # standardize features so the 3 very differently-scaled statistics are penalized fairly
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # --- sample weighting added here ---
    # Weight each training gene by how many cells its own fingerprint was estimated from.
    # Why: a gene's feature vector is a single fixed number per (gene, condition) -- it doesn't
    # vary from cell to cell -- so we can't genuinely train on individual cells here (there is no
    # per-cell version of "this gene's control dropout rate" to train on). But fitting Ridge on
    # each gene's cells repeated one row per cell (with the same repeated X) is mathematically
    # identical to fitting once per gene with sample_weight = number of cells (this follows from
    # how squared-error loss decomposes over repeated rows). So this line gives us that same
    # effect directly, without actually duplicating any rows: genes whose fingerprint rests on
    # more cells (so is less noisy) get to pull the fit harder than genes backed by only ~100 cells.
    sample_weight = cell_counts.loc[fit_genes, condition].values

    ridge = Ridge(alpha=alpha)
    ridge.fit(X_scaled, y, sample_weight=sample_weight)

    query_X = scaler.transform(gene_features[(query_gene, condition)].reshape(1, -1))
    pred_scores = ridge.predict(query_X)

    # reconstruct the full fingerprint from the predicted target-PC scores
    pred_fingerprint = target_pca_by_condition[condition].inverse_transform(pred_scores)[0]
    return pred_fingerprint

## Choosing the Ridge regularization strength (alpha) via leave-one-out cross-validation

Hold out one training gene at a time, predict it from the other 39 (per condition), and compare a few candidate alpha values. This never touches the 10 held-out test genes -- alpha is fixed before we ever look at them.

In [6]:
candidate_alphas = [0.1, 1, 10, 100, 1000, 10_000, 100_000, 1_000_000]

cv_mse_by_alpha = {}
for alpha in candidate_alphas:
    squared_errors = []
    for cond in conditions:
        for gene in train_40:
            # ridge_predict excludes the query gene itself from the pool, so this is a genuine leave-one-out prediction
            pred = ridge_predict(gene, cond, alpha, train_40)
            true = pert_FC_selected.loc[(gene, cond)].values
            squared_errors.append(np.mean((true - pred) ** 2))
    cv_mse_by_alpha[alpha] = np.mean(squared_errors)

best_alpha = min(cv_mse_by_alpha, key=cv_mse_by_alpha.get)
cv_mse_by_alpha, best_alpha

({0.1: np.float64(0.002746818929116736),
  1: np.float64(0.002740672119055408),
  10: np.float64(0.0027125859068649312),
  100: np.float64(0.0026682478051749375),
  1000: np.float64(0.002567986231998191),
  10000: np.float64(0.0023144296671500243),
  100000: np.float64(0.002204073324818075),
  1000000: np.float64(0.002194835951647649)},
 1000000)

## Predict the held-out test genes and evaluate

Use the chosen alpha to predict each of the 10 held-out genes from the 40 training genes, then evaluate with the same metrics used for the baseline and k-NN models so results are directly comparable.

In [7]:
# predict each held-out (gene, condition) pair from the 40 training genes, using the CV-chosen alpha
ridge_predictions = {
    (gene, cond): ridge_predict(gene, cond, best_alpha, train_40)
    for cond in conditions
    for gene in test_10
}


def evaluate_predictions(true_df, predictions_by_row):
    """Compare each true fingerprint against its predicted fingerprint (looked up per row)."""
    records = []
    for (pert, cond), true_fc in true_df.iterrows():
        pred_fc = predictions_by_row[(pert, cond)]
        pearson_r, _ = pearsonr(true_fc, pred_fc)
        spearman_r, _ = spearmanr(true_fc, pred_fc)
        mse = np.mean((true_fc - pred_fc) ** 2)
        records.append({
            "perturbation": pert,
            "condition": cond,
            "pearson_r": pearson_r,
            "spearman_r": spearman_r,
            "mse": mse,
        })
    return pd.DataFrame(records)


ridge_eval = evaluate_predictions(pert_FC_selected.loc[test_10, :], ridge_predictions)
ridge_eval

,perturbation,condition,pearson_r,spearman_r,mse
0,KCNN4,Control,0.830469,0.456849,0.001492
1,KCNN4,IFNγ,0.845380,0.393048,0.001085
2,KCNN4,Co-culture,0.850624,0.345230,0.001183
3,TIMM50,Control,0.741452,0.393119,0.002769
4,TIMM50,IFNγ,0.623775,0.294416,0.003300
5,TIMM50,Co-culture,0.688057,0.184655,0.003901
6,TXNDC17,Control,0.803929,0.498089,0.004381
7,TXNDC17,IFNγ,0.624036,0.447916,0.004287
8,TXNDC17,Co-culture,0.775137,0.407479,0.004350
9,CORO1A,Control,0.808652,0.373048,0.001173


In [8]:
metrics = ["pearson_r", "spearman_r", "mse"]

# per-condition breakdown (n=10 genes each) -- for biological interpretation
per_condition = ridge_eval.groupby("condition")[metrics].agg(["mean", "std"])

# pooled across all held-out (gene, condition) pairs (n=30) -- single headline number, comparable to the other models
overall = ridge_eval[metrics].agg(["mean", "std"])

per_condition

pearson_r           spearman_r                 mse          
                mean       std       mean       std      mean       std
condition                                                              
Co-culture  0.588823  0.497317   0.269515  0.110518  0.003995  0.005922
Control     0.756075  0.136345   0.399062  0.120588  0.002487  0.001584
IFNγ        0.718606  0.254092   0.378587  0.098350  0.002410  0.001898

In [9]:
overall

,pearson_r,spearman_r,mse
mean,0.687835,0.349055,0.002964
std,0.328442,0.121037,0.003651


## Discussion (initial draft -- please rewrite)

**What this notebook does:** Trains a Ridge regression model using genuinely gene-specific features: each of the 50 target genes' own expression statistics (mean, variance, dropout rate) computed from control cells, per condition. To keep the regression well-posed with only 40 training examples, the ~2042-dim RNA fingerprint target is reduced via PCA (fit only on the 40 training genes, per condition) to 10 components. Ridge maps the 3 features to the 10 target-PC scores; alpha is chosen via leave-one-gene-out cross-validation restricted to the 40 training genes; the fit is weighted by each gene's own cell count (`sample_weight`), so fingerprints estimated from more cells (and therefore less noisy) count for more.

**Results:** LOOCV never finds an interior minimum -- validation MSE decreases monotonically all the way out to alpha = 1,000,000, the largest value tried. That means the cross-validation procedure prefers *as much regularization as possible*, which drives Ridge's coefficients toward zero and makes its predictions collapse to (essentially) the training-mean fingerprint, weighted by cell count. This shows up directly in the final test numbers: Pearson r = 0.688, Spearman r = 0.349, MSE = 0.00296 -- statistically indistinguishable from the plain baseline in Task3_02.

**Interpretation:** a gene's own baseline expression level, variance, and dropout rate in *unperturbed* cells simply don't carry usable information about how strongly *that gene's own knockout* will reshape the transcriptome. This is a meaningful (if negative) result: it rules out "how highly/reliably a gene is normally expressed" as a useful predictor of perturbation effect size on its own, and motivates looking for features that are actually informed by some version of the perturbation itself (which is exactly what model 3 does with protein data).